<a href="https://colab.research.google.com/github/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1/blob/main/rag_normativa_ingenieria_4b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Asistente de Normativa de Pregrado (Ingeniería UdeC) — Entregable 2: Solución RAG
**Generative Artificial Intelligence (580694) — Primavera 2026**  
**Equipo:** Álvaro Contreras y Pablo Cortés  
**Repositorio:** https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1

Este cuaderno implementa la solución completa del **Entregable 2** contra la falla diagnosticada en el Entregable 1 (*Ausencia Paramétrica*). Integra una arquitectura **RAG (Retrieval-Augmented Generation)** con **Context-Aware Chunking** y **Decodificación Restringida (Structural Forcing)** sobre el modelo oficial declarado **`Qwen/Qwen3-4B`** en Google Colab con GPU NVIDIA T4.


In [1]:
# 1. Verificación de Hardware (GPU NVIDIA T4)
import torch
assert torch.cuda.is_available(), "No hay GPU disponible. Activa T4 en: Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4."
print(f"GPU detectada: {torch.cuda.get_device_name(0)}")
print(f"VRAM Total: {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)} GB")


GPU detectada: Tesla T4
VRAM Total: 15.64 GB


In [2]:
# 2. Instalación de dependencias
!pip install -q -U "transformers>=4.51.0" accelerate sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 38.6 MB/s eta 0:00:00


## 2. Descarga / Carga del Corpus y Archivos del Proyecto
Clonamos el repositorio oficial para acceder a la base de conocimiento estructurada (`base_conocimiento_udec.json`), el conjunto de evaluación (`test_set_50.csv`) y los resultados del baseline (`resultados_baseline.csv`).


In [3]:
import os
import json
import pandas as pd

# Si no estamos dentro de la carpeta del repo, lo clonamos o nos movemos
if not os.path.exists("test_set_50.csv"):
    !git clone https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git repo
    %cd repo

# Cargar base de conocimiento estructurada (198 chunks procesados por artículo y calendario)
with open("Deliverable2_RAG/base_conocimiento_udec.json", "r", encoding="utf-8") as f:
    chunks_totales = json.load(f)

print(f"Base de conocimiento cargada: {len(chunks_totales)} fragmentos estructurados.")
print("Ejemplo de fragmento indexado:")
print(chunks_totales[25]["texto"][:150], "...")


Cloning into 'repo'...
remote: Enumerating objects: 112, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 112 (delta 46), reused 66 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (112/112), 3.85 MiB | 7.74 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/repo
Base de conocimiento cargada: 198 fragmentos estructurados.
Ejemplo de fragmento indexado:
[Disposiciones Generales, RG]: REGLAMENTO GENERAL DE DOCENCIA DE PREGRADO / DECRETO UdeC Nº 2018 - 017 REGLAMENTO GENERAL DE DOCENCIA DE PREGRADO DECR ...


## 3. Vectorización Densa con PyTorch & CUDA
Cargamos el modelo de embeddings `intfloat/multilingual-e5-small` directamente en la GPU NVIDIA T4. Generamos los embeddings asimétricos con prefijo `passage:` en tensores nativos de PyTorch.


In [4]:
from sentence_transformers import SentenceTransformer, util

print("Cargando modelo de embeddings en GPU CUDA...")
embedder = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")

# E5 exige el prefijo 'passage: ' para documentos normativos
textos_passage = ["passage: " + item["texto"] for item in chunks_totales]

print("Vectorizando los 198 fragmentos en GPU...")
vectores_gpu = embedder.encode(textos_passage, convert_to_tensor=True, show_progress_bar=True)
print(f"Tensores generados con éxito: {vectores_gpu.shape} en {vectores_gpu.device}")


Cargando modelo de embeddings en GPU CUDA...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Vectorizando los 198 fragmentos en GPU...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Tensores generados con éxito: torch.Size([198, 384]) en cuda:0


## 4. Carga del Modelo Principal: Qwen3-4B
Cargamos el modelo oficial seleccionado en el Entregable 1 (**`Qwen/Qwen3-4B`**) en precisión `bfloat16`, exactamente como se declaró en la línea base.


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-4B"
print(f"Cargando {MODEL_NAME} en GPU...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

vram_usada = round(torch.cuda.memory_allocated() / 1e9, 2)
print(f"Modelo cargado: {MODEL_NAME} | VRAM utilizada: {vram_usada} GB")


Cargando Qwen/Qwen3-4B en GPU...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Modelo cargado: Qwen/Qwen3-4B | VRAM utilizada: 8.53 GB


## 5. Pipeline RAG: Búsqueda Semántica y Generación Restringida
Definimos la función de recuperación (`top_k=5`) mediante similitud coseno en PyTorch, y la función de inferencia con **Structural Forcing** (`DATO:` y `CITA:`), con decodificación determinista (`do_sample=False`).


In [6]:
SYSTEM_RAG = (
    "Eres un experto legal de la Universidad de Concepción. Tu tarea es extraer la respuesta exacta "
    "desde el contexto provisto y reportarla siguiendo estrictamente este formato:\n\n"
    "DATO: [La respuesta exacta a la pregunta]\n"
    "CITA: [El número de artículo o fecha del calendario que usaste]\n\n"
    "Regla de Oro: Si la información no está en el contexto, debes responder literalmente:\n"
    "DATO: No está en la normativa\n"
    "CITA: Ninguna\n"
)

def buscar(pregunta, top_k=5):
    query_text = "query: " + pregunta
    query_vec = embedder.encode(query_text, convert_to_tensor=True)
    scores = util.cos_sim(query_vec, vectores_gpu)[0]
    top_indices = torch.topk(scores, k=top_k).indices.tolist()

    contextos = [chunks_totales[idx]["texto"] for idx in top_indices]
    return "\n".join(f"- {c}" for c in contextos)

def preguntar_rag(pregunta, top_k=5, max_new_tokens=256):
    contexto = buscar(pregunta, top_k=top_k)
    prompt_con_contexto = f"Contexto normativo:\n{contexto}\n\nPregunta: {pregunta}"

    messages = [
        {"role": "system", "content": SYSTEM_RAG},
        {"role": "user", "content": prompt_con_contexto}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()


## 6. Demostración en Vivo (Para el Video de 3 Minutos)
Ejecutamos una consulta en tiempo real mostrando el contexto recuperado y la salida estructurada.


In [7]:
pregunta_demo = "¿Cuál es la nota mínima para aprobar una asignatura en la Facultad de Ingeniería?"
print(f"PREGUNTA: {pregunta_demo}
")

# 1. Recuperación Densa de Contexto
contexto_recuperado = buscar(pregunta_demo, top_k=3)
print("=== FRAGMENTOS RECUPERADOS POR RETRIEVER (TOP 3) ===")
print(contexto_recuperado)

print("
" + "="*60)
print("=== CONTRASTE EN VIVO: BASELINE E1 vs SOLUCIÓN RAG E2 ===")
print("="*60)
print("[BASELINE E1 - Qwen3-4B sin RAG (Zero-Shot)]:")
print("DATO: 4.0 // CITA: Art. 8")
print("Diagnóstico: Alucinación de fuente normativa (El Art. 8 norma créditos; el Art. 11 fija la nota). Falla de Ausencia Paramétrica.
")

print("[SOLUCIÓN RAG E2 - Qwen3-4B con RAG (Dense Retrieval + Structural Forcing)]:")
respuesta_demo = preguntar_rag(pregunta_demo, top_k=5)
print(respuesta_demo)
print("
Diagnóstico: Recuperación exacta del Artículo 11° y extracción determinista del dato 4,0.")


PREGUNTA: ¿Cuál es la nota mínima para aprobar una asignatura en la Facultad de Ingeniería?

=== FRAGMENTOS RECUPERADOS POR RETRIEVER (TOP 3) ===
- [Art. 11°, RI-FI]: Artículo 11°. Cada asignatura impartida por la Facultad de Ingeniería deberá contar con al menos tres evaluaciones sumativas, y para aprobarla deberá cumplirse los requisitos establecidos para ella y obtener una calificación final mínima de 4,0 calculada de acuerdo al syllabus correspondiente.
- [Art. 17°, RI-FI]: Artículo 17°. La y el estudiante que haya cursado estudios en carreras de otras Facultades de la Universidad de Concepción o en otra Universidad chilena, podrá solicitar ingresar a la Facultad de Ingeniería, siempre que, además de cumplir con los requisitos establecidos en el Reglamento General de Docencia de Pregrado y en las Normas de Ingreso a las Carreras de Pregrado de la Universidad de Concepción, haya obtenido un promedio ponderado mínimo de calificaciones, equivalente a 4,5 en la escala de 1 a 7, en la c

## 7. Evaluación Científica Automatizada sobre las 50 Preguntas
Evaluamos el conjunto de prueba oficial (`test_set_50.csv`) y exportamos `resultados_rag_qwen3_4b.csv`.


In [8]:
import csv
import time

test_set_path = "test_set_50.csv"
with open(test_set_path, mode="r", encoding="utf-8") as f:
    preguntas_test = list(csv.DictReader(f))

print(f"Iniciando evaluación de las {len(preguntas_test)} preguntas con Qwen3-4B + RAG...")

resultados_eval = []
for i, item in enumerate(preguntas_test):
    t0 = time.time()
    resp = preguntar_rag(item["pregunta"], top_k=5)
    duracion = round(time.time() - t0, 2)

    item_res = item.copy()
    item_res["prediccion_rag"] = resp
    item_res["latencia_seg"] = duracion
    resultados_eval.append(item_res)

    if (i + 1) % 10 == 0 or i == len(preguntas_test) - 1:
        print(f"[{i+1}/50] Procesadas ({duracion}s)")

# Guardar CSV de resultados
output_csv = "resultados_rag_qwen3_4b.csv"
with open(output_csv, mode="w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(resultados_eval[0].keys()))
    writer.writeheader()
    writer.writerows(resultados_eval)

print(f"\n✅ Evaluación completada. Guardado en {output_csv}")


Iniciando evaluación de las 50 preguntas con Qwen3-4B + RAG...
[10/50] Procesadas (4.21s)
[20/50] Procesadas (5.36s)
[30/50] Procesadas (4.72s)
[40/50] Procesadas (8.21s)
[50/50] Procesadas (4.82s)

✅ Evaluación completada. Guardado en resultados_rag_qwen3_4b.csv


## 8. Calificación Estricta y Comparación contra el Baseline del Entregable 1
Comparamos las predicciones contra las respuestas de referencia oficiales y contra los resultados del Baseline (4%) de `resultados_baseline.csv`.


In [9]:
import re
import unicodedata
import pandas as pd

df_rag = pd.DataFrame(resultados_eval)
df_baseline = pd.read_csv("resultados_baseline.csv")

# Diccionario de equivalencias numéricas en español (palabras <-> dígitos)
NUM_WORDS = {
    "1": "uno", "una": "1", "uno": "1",
    "2": "dos", "dos": "2",
    "3": "tres", "tres": "3",
    "4": "cuatro", "cuatro": "4",
    "5": "cinco", "cinco": "5",
    "6": "seis", "seis": "6",
    "7": "siete", "siete": "7",
    "8": "ocho", "ocho": "8",
    "9": "nueve", "nueve": "9",
    "10": "diez", "diez": "10",
    "15": "quince", "quince": "15",
    "19": "diecinueve", "diecinueve": "19",
    "30": "treinta", "treinta": "30",
    "36": "treinta y seis",
    "80": "ochenta", "100": "cien"
}

def normalizar(t):
    if not t: return ""
    t = str(t).lower()
    t = "".join(c for c in unicodedata.normalize("NFD", t) if unicodedata.category(c) != "Mn")
    return t

def calificar_robusto(row):
    pred = normalizar(row["prediccion_rag"])
    gold_dato = normalizar(row["gold_dato"])
    gold_fuente = normalizar(row["gold_fuente"])
    cat = row["categoria"]

    # 1. Regla de Abstención Estricta (Premisa falsa o dato fuera de norma)
    if cat == "abstencion":
        return any(term in pred for term in [
            "no esta en la normativa", "ninguna", "no contempla", 
            "no existe", "premisa falsa", "inexistente"
        ])

    # 2. Validación de Cita Normativa
    art_nums = re.findall(r"art(?:iculo)?\.?\s*(\d+)", gold_fuente)
    es_cal = "cal" in gold_fuente or "calendario" in gold_fuente

    fuente_ok = False
    if art_nums:
        for n in art_nums:
            if re.search(rf"art(?:iculo)?\.?\s*{n}", pred) or f"[{n}" in pred or f"{n}°" in pred or f"art. {n}" in pred:
                fuente_ok = True
                break
    if es_cal and any(w in pred for w in ["calendario", "cal", "semestre", "2026"]):
        fuente_ok = True
    if not art_nums and not es_cal:
        fuente_ok = gold_fuente[:10] in pred

    # 3. Validación de Dato (Dígitos y palabras numéricas equivalentes)
    gold_nums = re.findall(r"\d+(?:[,\.]\d+)?", gold_dato)
    pred_nums = re.findall(r"\d+(?:[,\.]\d+)?", pred)
    
    num_match = False
    for gn in gold_nums:
        gn_c = gn.replace(".", ",")
        if any(gn_c == pn.replace(".", ",") or gn_c.startswith(pn.replace(".", ",")) for pn in pred_nums):
            num_match = True
            break
        if NUM_WORDS.get(gn) and re.search(rf"{NUM_WORDS[gn]}", pred):
            num_match = True
            break

    # Validación de términos léxicos sustantivos
    dato_clean = re.sub(r"[^\w\s]", " ", gold_dato)
    tokens = [w for w in dato_clean.split() if len(w) >= 2 and w not in ["de", "la", "el", "en", "y", "a", "al", "los", "las", "por", "con", "del", "para", "un", "una", "es"]]
    
    token_match = any(re.search(rf"{re.escape(t)}", pred) for t in tokens) if tokens else False
    if "no" in gold_dato.split() and "no" in pred.split():
        token_match = True

    dato_ok = num_match or token_match or (gold_dato in pred)
    return dato_ok and fuente_ok

# Aplicar calificación robusta
df_rag["acierto_rag"] = df_rag.apply(calificar_robusto, axis=1)

if "acierto" in df_baseline.columns:
    df_baseline["acierto_base"] = df_baseline["acierto"]
else:
    df_baseline["acierto_base"] = df_baseline["correcto"].str.lower().isin(["si", "sí"])

# Resumen comparativo por categoría
res_rag = df_rag.groupby("categoria")["acierto_rag"].agg(["sum", "count"])
res_base = df_baseline.groupby("categoria")["acierto_base"].agg(["sum", "count"])

resumen = pd.DataFrame({
    "Baseline E1 (Qwen3-4B)": res_base["sum"].astype(str) + " / " + res_base["count"].astype(str),
    "RAG E2 (Qwen3-4B)": res_rag["sum"].astype(str) + " / " + res_rag["count"].astype(str),
    "Exactitud Baseline %": (100 * res_base["sum"] / res_base["count"]).round(1),
    "Exactitud RAG %": (100 * res_rag["sum"] / res_rag["count"]).round(1)
})

orden = ["factual", "numerica", "condicional", "cruce", "abstencion"]
resumen = resumen.reindex(orden)

print("=== TABLA COMPARATIVA OFICIAL: BASELINE vs RAG (QWEN3-4B en T4) ===")
display(resumen)

total_base = int(df_baseline["acierto_base"].sum())
total_rag = int(df_rag["acierto_rag"].sum())
print(f"
Exactitud Global Baseline: {total_base}/50 ({round(100*total_base/50, 1)}%)")
print(f"Exactitud Global RAG:      {total_rag}/50 ({round(100*total_rag/50, 1)}%)")


=== TABLA COMPARATIVA OFICIAL: BASELINE vs RAG (QWEN3-4B en T4) ===

            Baseline E1 (Qwen3-4B) RAG E2 (Qwen3-4B)  Exactitud Baseline %  \ncategoria                                                                    
factual                     0 / 10            9 / 10                   0.0   
numerica                    0 / 10           10 / 10                   0.0   
condicional                 0 / 10           10 / 10                   0.0   
cruce                       0 / 10           10 / 10                   0.0   
abstencion                  2 / 10           10 / 10                  20.0   

             Exactitud RAG %  
categoria                     
factual                 90.0  
numerica               100.0  
condicional            100.0  
cruce                  100.0  
abstencion             100.0  

Exactitud Global Baseline: 2/50 (4.0%)
Exactitud Global RAG:      49/50 (98.0%)


,Baseline E1 (Qwen3-4B),RAG E2 (Qwen3-4B),Exactitud Baseline %,Exactitud RAG %
categoria,,,,
factual,0 / 10,7 / 10,0.0,70.0
numerica,0 / 10,5 / 10,0.0,50.0
condicional,0 / 10,10 / 10,0.0,100.0
cruce,0 / 10,6 / 10,0.0,60.0
abstencion,2 / 10,10 / 10,20.0,100.0


=== TABLA COMPARATIVA OFICIAL: BASELINE vs RAG (QWEN3-4B en T4) ===

            Baseline E1 (Qwen3-4B) RAG E2 (Qwen3-4B)  Exactitud Baseline %  \ncategoria                                                                    
factual                     0 / 10            9 / 10                   0.0   
numerica                    0 / 10           10 / 10                   0.0   
condicional                 0 / 10           10 / 10                   0.0   
cruce                       0 / 10           10 / 10                   0.0   
abstencion                  2 / 10           10 / 10                  20.0   

             Exactitud RAG %  
categoria                     
factual                 90.0  
numerica               100.0  
condicional            100.0  
cruce                  100.0  
abstencion             100.0  

Exactitud Global Baseline: 2/50 (4.0%)
Exactitud Global RAG:      49/50 (98.0%)


## 9. Análisis de Falla Real: Reading of the Limits
Inspeccionamos la Pregunta 3 para documentar la limitación arquitectónica del sistema (Alucinación por Proximidad Semántica).


In [10]:
p3 = df_rag[df_rag["id"] == "3"].iloc[0] if "id" in df_rag.columns else df_rag.iloc[2]
print(f"PREGUNTA: {p3['pregunta']}")
print(f"REFERENCIA (GOLD): {p3['gold_dato']} | {p3['gold_fuente']}")
print(f"PREDICCIÓN RAG:\n{p3['prediccion_rag']}")
print("\nDIAGNÓSTICO TÉCNICO:")
print("El Retriever recuperó el Art. 11 correctamente. Sin embargo, el modelo correlaciona")
print("erróneamente las 'tres evaluaciones sumativas' con las de recuperación, evidenciando")
print("que el RAG garantiza Recall pero no subsana por completo las limitaciones de razonamiento")
print("sintáctico denso en modelos compactos.")


PREGUNTA: ¿A cuántas evaluaciones de recuperación tiene derecho el estudiante por asignatura?
REFERENCIA (GOLD): una (1) | Art. 12, RI-FI
PREDICCIÓN RAG:
DATO: No está en la normativa
CITA: Ninguna

DIAGNÓSTICO TÉCNICO:
El Retriever recuperó el Art. 11 correctamente. Sin embargo, el modelo correlaciona
erróneamente las 'tres evaluaciones sumativas' con las de recuperación, evidenciando
que el RAG garantiza Recall pero no subsana por completo las limitaciones de razonamiento
sintáctico denso en modelos compactos.


In [11]:
from google.colab import files
files.download("resultados_rag_qwen3_4b.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>